# الدرس الثاني: التكامل الموحد لنماذج اللغة عبر init_chat_model

## المقدمة والاهداف التعليمية
في هذا الدفتر، سنتعرف على الدالة الموحدة `init_chat_model` التي اطلقتها LangChain لحل مشكلة تعدد فئات النماذج وتوحيد واجهة الاستدعاء عبر مختلف مزودي خدمات الذكاء الاصطناعي.

## ما المشكلة التي حلتها `init_chat_model`؟
في الاصدارات السابقة، كان المطور مضطرا لاستيراد فئة مخصصة لكل مزود:
- `from langchain_openai import ChatOpenAI`
- `from langchain_anthropic import ChatAnthropic`
- `from langchain_groq import ChatGroq`
- `from langchain_google_genai import ChatGoogleGenerativeAI`

هذا الاسلوب كان يجعل التبديل بين النماذج يتطلب تعديلا شاملا في الكود (Refactoring). مع `init_chat_model`:
1. يتم تحديد اسم النموذج واسم المزود ديناميكيا عبر متغيرات نصية.
2. تتوحد جميع معلمات التهيئة مثل `temperature`, `max_tokens`, `model_kwargs`.
3. تتيح التطبيقات التبديل المرن بين النماذج دون تعديل بنية الشيفرة البرمجية.

## الخطوة 1: تحميل المتغيرات البيئية واستيراد الدالة الموحدة
نقوم بتحميل المتغيرات البيئية واستيراد `init_chat_model` من حزمة `langchain.chat_models`.

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")

## الخطوة 2: تهيئة النموذج عبر مزود Groq
نقوم بتمرير اسم النموذج واسم المزود (`model_provider="groq"`) واستدعاء النموذج بالاستعلام المباشر.

In [ ]:
# تهيئة نموذج سريع عبر Groq
groq_model = init_chat_model(
    "openai/gpt-oss-120b",
    model_provider="groq",
    temperature=0.2
)

# استدعاء النموذج
response = groq_model.invoke("What is the capital of France and what is its population?")
print("Provider:", groq_model)
print("Response Content:")
print(response.content)

## الخطوة 3: فحص بيانات المخرجات وبيانات التوكنات (Metadata & Token Usage)
توفر الكائنات المرجعة من `init_chat_model` بيانات تفصيلية موحدة حول عدد الرموز المستهلكة (Tokens) وسبب انتهاء التوليد (Finish Reason).

In [ ]:
print("Response Metadata:")
for key, value in response.response_metadata.items():
    print(f"  {key}: {value}")

if hasattr(response, "usage_metadata") and response.usage_metadata:
    print("\nUsage Statistics:")
    for metric, count in response.usage_metadata.items():
        print(f"  {metric}: {count}")

## الخطوة 4: مثال توضيحي للتبديل المرن بين المزودين
توضح الدالة التالية كيف يمكن للتطبيق دعم عدة مزودين عبر معيار برمجي موحد دون كتابة شروط معقدة (if-else).

In [ ]:
def get_chat_model(provider_name: str, model_id: str):
    """Universal factory function using init_chat_model."""
    return init_chat_model(
        model=model_id,
        model_provider=provider_name,
        temperature=0.1
    )

# امثلة نظرية لكيفية استدعاء مزودين مختلفين بنفس الواجهة
print("Factory configuration ready:")
print("1. Groq: init_chat_model('openai/gpt-oss-120b', model_provider='groq')")
print("2. OpenAI: init_chat_model('gpt-4o', model_provider='openai')")
print("3. Anthropic: init_chat_model('claude-3-5-sonnet-20241022', model_provider='anthropic')")
print("4. Google: init_chat_model('gemini-1.5-pro', model_provider='google_genai')")